In [462]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [463]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q304227,Aba,NaN,ninfa naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
2,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmente una ninfa amata da Apollo",greek_deity


In [464]:
#estraggo tutte le label  
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto 
myth_terms = list(set(labels + aliases)) #unisco le label e gli alias, tolgo i duplicati con set e riconverto in lista
len(myth_terms)

3121

In [465]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [466]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2258640


In [468]:
#query pr estrarre i dati dei beni storico artistici in batch 
# Iterare su tutti i 2.258.640 beni con batch di 20000

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

URLError: <urlopen error [WinError 10054] Connessione in corso interrotta forzatamente dall'host remoto>

In [ ]:
# i rows sono già stati accumulati dal loop nel batch precedente
# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(1)

Righe totali in df_arco : 1120195


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704280,"Arlecchino primordiale. Fine del cinquecento. Martinelli, figurino per costume di scena, maschile (disegno) by Fo, Dario (XX)","figurino per costume di scena, maschile"


In [ ]:
import re

# 1. filtro base (toglie null)
df_arco = df_arco[df_arco["subject"].notna()].copy()

#TERMINI DA ESCLUDERE TOTALMENTE  
ignore_subjects = [
    "Acheo","Africa","Agatone","Aia","Aix","Alessandro","Alessi","Angelo","Aniceto","Anna","Antenore","Antioco","Api","Arche","Archelao","Armeno","Astarte","Asteria","Asterio","Asia","Argia","Astaco","Atteo","Aula","Aventino","Basile","Bolina","Bronte","Calice","Camilla", "Canto","Capeto","Car",
    "Cassandra","Cerano", "Cino","Cleopatra","Cocito","Como", "Copia","Corinto","Coro","Creta", "Damaso","Delfino", "Egisto","Egitto","Egle","Elea", "Eleuteria", "Eleutero", "Elia","Elios","Emo", "Epulone","Erme","Eschilo","Esione","Etna", "Etolo", "Ezio", "Fante", "Fenice", "Festo", "Finta",
    "Giacinto", "Giano", "Grillo", "Greco", "Idra", "Ippolita", "Leo"

]


df_arco = df_arco[
    ~df_arco["subject"].str.contains(
        "|".join(ignore_subjects),
        case=False,
        na=False
    )
]


terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()
]

# regex unica case-sensitive (matching esatto)
pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(terms) + r')(?![A-Za-zÀ-ÿ-])'
regex = re.compile(pattern)


#TERMINI DA ACCETTARE SOLO SE COMPAIONO INSIEME A QUESTE PAROLE []
context_rules = {
    "Ascanio" : ["fuoco", "cervo"],
    "Callisto": ["Arcade", "Diana"],
    "Concordia": ["Allegoria", "Innocenza", "Giustizia"],
    "Cornacchia": ["metamorfosi"],
    "Egeo": ["Teseo"],
    "Elena": ["ratto"],
    "Ettore": ["morte", "Protesilao", "Achille", "Andromaca", "Paride"],
    "Europa": ["ratto", "Toro", "sibilla", "mito","allegoria","ancelle","leone"],
    "Grazie" : ["tre", "danzanti"],
    "Ida" : ["Linceo"],
    "Ippolito":["episodi", "caduta", "morte"]
}


# TERMINI DA ESCLUDERE SE COMPAIONO CON DETERMINATE PAROLE
exclusion_rules = {
    "Abbondanza": ["Santa", "Sant"],
    "Amore": ["Dio", "Madonna", "castello", "fontana"],
    "Carità": ["Santa", "Sant", "Maria"],
    "Diana" : ["Beata", "duca"],
    "Dioniso": ["Vescovo"],
    "Elio" : ["ponte","ritratto","paesaggio", "veduta"],
    "Enea": ["Beato", "discepolo"],
    "Ercole": ["Ricotti", "Dandini", "Comini", "Farnese"],
    "Ermes" : ["ritratto"],
    "Fede" : ["santi", "sant", "san", "dottrina", "biblici", "Francia", "cattolica", "madonna", "evangelisti", "religione", "dio", "angeli", "angioletti","santa", "papa", "santo", "cristiana"],
    "Giustizia" : ["porta", "palazzo", "misericordia", "san", "santa", "papa", "imperatore", "papale"],
    "Gorgone" : ["santa"],
    "Io": ["ritratti "],
    "Lavinia":["ritratto", "autoritratto"],
    "Leandro" : ["autoritratto", "San", "busto", "torre"]

}


def validate_matches(row):

    subject = row["subject"]

    valid_matches = []

    for match in row["matched_term"]:

        # PRIMO CHECK: controlla se il termine è nelle exclusion_rules
        if match in exclusion_rules:
            exclusion_keywords = exclusion_rules[match]
            # se almeno una parola di esclusione è presente, scarta il termine
            if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in exclusion_keywords):
                continue  # salta questo match, non lo aggiungere a valid_matches
        
        # SECONDO CHECK: controlla context_rules (inclusione condizionata)
        # se il termine non ha regole → tienilo
        if match not in context_rules:
            valid_matches.append(match)
            continue

        # controlla parole contestuali con word boundaries (parole intere)
        keywords = context_rules[match]

        if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in keywords):
            valid_matches.append(match)

    return valid_matches


df_arco["matched_term"] = df_arco["subject"].str.findall(regex)
# matching veloce
df_arco["matched_term"] = df_arco.apply(validate_matches, axis=1)

# solo match
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]



In [ ]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 460102
Beni con match mitologico: 11551


In [ ]:
pd.set_option("display.max_colwidth", None)

start = 5200
end = 5400

#SONO ARRIVATA A "LIDO"

df_matches_sorted = df_matches.sort_values("matched_term")

df_matches_sorted[["item","label", "subject", "matched_term"]].iloc[start:end]


,item,label,subject,matched_term
1797516,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0300182761,"Leda e il cigno (disegno, opera isolata) di Cosway Richard (sec. XIX)",Leda e il cigno,[Leda e il cigno]
1797628,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0300182817,"Leda e il cigno (disegno, opera isolata) di Cosway Richard (sec. XIX)",Leda e il cigno,[Leda e il cigno]
2007322,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900768831,"Leda, Leda e il cigno (statuetta, opera isolata) di Cassioli Giovanni (attribuito) (sec. XX)",Leda e il cigno,[Leda e il cigno]
2205476,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0300140197,"Leda e il cigno (dipinto, opera isolata) - ambito mantovano (seconda metà sec. XVIII)",Leda e il cigno,[Leda e il cigno]
355538,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900281624-7,Leda e il cigno (dipinto) by Tommaso di Battista del Verrocchio (sec. XVI),Leda e il cigno,[Leda e il cigno]
475888,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900281646-1_13,Leda e il cigno (dipinto) by Marchetti Marco detto Marco da Faenza (sec. XVI),Leda e il cigno,[Leda e il cigno]
2061572,https://w3id.org/arco/resource/Lombardia/HistoricOrArtisticProperty/P0120-00231_R03,Leda e il cigno (disegno) by Gallina Gallo (secondo quarto sec. XIX),Leda e il cigno,[Leda e il cigno]
678792,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0800285522,Leda e il cigno (gemma) - produzione romana (sec. I a.C),Leda e il cigno,[Leda e il cigno]
601896,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1500334144,Leda e il cigno (disegno) - ambito italiano (sec. XIX),Leda e il cigno,[Leda e il cigno]
355862,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900281662-3,Leda e il cigno (dipinto) by Marchetti Marco detto Marco da Faenza (sec. XVI),Leda e il cigno,[Leda e il cigno]
